# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**
**Student ID:**
**Group:**
**Repository:**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. | 9 |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. | 10 |
| D3 | Embeddings represent text as numerical vectors. | 6 |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl = (9 + 10 + 6) / 3 = 25/3 ≈ 8,333**

**Query terms: how, does, rag, retrieve, information**

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3,5/0,5) = 1,9459 |
| does | 0 | 0 | 0 | 0 | 1,9459 |
| rag | 0 | 0 | 0 | 0 | 1,9459 |
| retrieve | 0 | 0 | 0 | 0 | 1,9459 |
| information | 0 | 0 | 0 | 0 | 1,9459 |

**BM25 of each document**: Normalised length factor: `k1 · (1 − b + b · |d| / avgdl)` with k1 = 1.5, b = 0.75, avgdl = 25/3 ≈ 8.333

- D1: 1,5 · (0,25 + 0,75 · 9/8,333) = 1,5 · 1,06 = 1,59
- D2: 1,5 · (0,25 + 0,75 · 10/8,333) = 1,5 · 1,15 = 1,725
- D3: 1,5 · (0,25 + 0,75 · 6/8,333) = 1,5 · 0,79 = 1,185

Every query term has f(t, d) = 0 in every document, so the numerator `idf(t) · f(t,d) · (k1 + 1)` is 0 for each term. No term contributes to any score.

**D1**
- how: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- does: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- rag: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- retrieve: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- information: 1,9459 · 0 · 2,5 / (0 + 1,59) = 0
- score(q, D1) = 0 + 0 + 0 + 0 + 0 = **0**

**D2**
- how: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- does: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- rag: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- retrieve: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- information: 1,9459 · 0 · 2,5 / (0 + 1,725) = 0
- score(q, D2) = 0 + 0 + 0 + 0 + 0 = **0**

**D3**
- how: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- does: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- rag: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- retrieve: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- information: 1,9459 · 0 · 2,5 / (0 + 1,185) = 0
- score(q, D3) = 0 + 0 + 0 + 0 + 0 = **0**

>

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tie) |
| D2 | 0 | 1 (tie) |
| D3 | 0 | 1 (tie) |

**Answer (a):** BM25 retrieves no document first: all three scores are 0, so the ranking is a three-way tie. I re-checked the term frequencies and the result holds. Without stemming, BM25 only matches identical tokens. D2 contains `retrieval` and `retrieves`, but not `retrieve`, and `rag`, `how`, `does` and `information` appear in none of the documents. So no query term matches any document token exactly.

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl = (9 + 10 + 6) / 3 = 25/3 ≈ 8.333 (stemming replaces tokens one-for-one, so the lengths do not change)**

**Stemmed query terms: how, doe, rag, retriev, inform**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3,5/0,5) = 1,9459 |
| doe | 0 | 0 | 0 | 0 | 1,9459 |
| rag | 0 | 0 | 0 | 0 | 1,9459 |
| retriev | 0 | 2 | 0 | 1 | ln(2,5/1,5) = 0,5108 |
| inform | 0 | 0 | 0 | 0 | 1,9459 |

**BM25 of each document**, worked:
Length factor `k1 · (1 − b + b · |d| / avgdl)`:

- D1: 1,5 · (0,25 + 0,75 · 9/8,333) = 1,59
- D2: 1,5 · (0,25 + 0,75 · 10/8,333) = 1,725
- D3: 1,5 · (0,25 + 0,75 · 6/8,333) = 1,185

**D1**
- how, doe, rag, inform: f = 0, so each contributes 0
- retriev: 0,5108 · 0 · 2,5 / (0 + 1,59) = 0
- score(q, D1) = **0**

**D2**
- how, doe, rag, inform: f = 0, so each contributes 0
- retriev: 0,5108 · 2 · 2,5 / (2 + 1,725) = 2,554 / 3,725 = 0,686
- score(q, D2) = 0 + 0 + 0 + 0,686 + 0 = **0,686**

**D3**
- how, doe, rag, inform: f = 0, so each contributes 0
- retriev: 0,5108 · 0 · 2,5 / (0 + 1,185) = 0
- score(q, D3) = **0**

>

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tie) |
| D2 | 0.686 | 1 |
| D3 | 0 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**
**(a) As written: no document.** All three documents score 0, so the ranking is a three-way tie and BM25 has no first result. I re-checked the term frequencies and the result holds, so it is not a calculation mistake. Without stemming, BM25 only counts tokens that are identical to a query term. D2 contains `retrieval` and `retrieves`, but the query word is `retrieve`, which is a different token. The other query terms (`how`, `does`, `rag`, `information`) do not appear in any document. Every f(t, d) is therefore 0, and every score is 0.

**(b) With stemming: D2.** Stemming maps `retrieval`, `retrieves` and `retrieve` to the same stem `retriev`, so the query term now matches D2, where it appears twice. D2 scores 0.686 and is the only document with a non-zero score. D1 and D3 still score 0 and remain tied for second place.

**Why:** BM25 is a lexical method. It scores only exact token matches, so a small difference in word form (`retrieve` vs `retrieves`) can decide whether a document is found at all. Stemming reduces that problem, but it cannot fix everything. `rag` is an acronym for "Retrieval-Augmented Generation", yet BM25 cannot connect it to the full phrase in D2, so that match is still missed.
>

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```
Formula: `cos(q, d) = (q · d) / (|q| · |d|)`

**Magnitude of the query:**

|q| = √(0,8² + 0,2² + 0,5²) = √(0,64 + 0,04 + 0,25) = √0,93 ≈ 0,9644

**D1**
- q · D1 = 0,8·0,7 + 0,2·0,1 + 0,5·0,4 = 0,56 + 0,02 + 0,20 = 0,78
- |D1| = √(0,49 + 0,01 + 0,16) = √0,66 ≈ 0,8124
- cos(q, D1) = 0,78 / (0,9644 · 0,8124) = 0,78 / 0,7835 ≈ **0,9956**

**D2**
- q · D2 = 0,8·0,9 + 0,2·0,3 + 0,5·0,6 = 0,72 + 0,06 + 0,30 = 1,08
- |D2| = √(0,81 + 0,09 + 0,36) = √1,26 ≈ 1,1225
- cos(q, D2) = 1,08 / (0,9644 · 1,1225) = 1,08 / 1,0825 ≈ **0,9977**

**D3**
- q · D3 = 0,8·0,1 + 0,2·0,8 + 0,5·0,2 = 0,08 + 0,16 + 0,10 = 0,34
- |D3| = √(0,01 + 0,64 + 0,04) = √0,69 ≈ 0,8307
- cos(q, D3) = 0,34 / (0,9644 · 0,8307) = 0,34 / 0,8011 ≈ **0,4244**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| D1 | 0,8·0,7 + 0,2·0,1 + 0,5·0,4 = 0,78 | 0,9644 | 0,8124 | 0,9956 |
| D2 | 0,8·0,9 + 0,2·0,3 + 0,5·0,6 = 1,08 | 0,9644 | 1,1225 | 0,9977 |
| D3 | 0,8·0,1 + 0,2·0,8 + 0,5·0,2 = 0,34 | 0,9644 | 0,8307 | 0,4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0,9956 | 2 |
| D2 | 0,9977 | 1 |
| D3 | 0,4244 | 3 |

**Which document is most similar to the query?**
D2, with cosine 0.9977. Its vector points in almost the same direction as the query vector. D1 is a very close second (0.9956), and D3 is far behind (0.4244) because its largest component is the second dimension, while the query's largest component is the first.
>

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.
Only partly. In A1 (a), BM25 gave no ranking at all: all three documents scored 0 and tied. In A1 (b), with stemming, BM25 ranks D2 first (0.686) and leaves D1 and D3 tied at 0. Cosine similarity agrees that D2 is first, but it also ranks D1 clearly above D3 (0.9956 vs 0.4244), a distinction BM25 cannot make in either version.

>

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.
BM25 only counts identical tokens, and this query shares almost none with the documents. `RAG` is an acronym, while D2 spells out "Retrieval-Augmented Generation". `retrieve` is not the same token as `retrieves` or `retrieval` unless we stem. `how`, `does` and `information` appear in no document. A dense embedding model maps text to vectors by meaning, so it can place the query near D2 (the document about retrieval and generation) even though few words match. It can also place D1 (about language models answering questions) closer to the query than D3 (about vectors), because D1 is topically related to the question being asked, even though it shares no exact word with the query.
>

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | | | |
| Q-10 | | | |

## Results for all ten or more queries: top 3 from each retriever

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | | | | |
| Q-02 | | | | |
| Q-03 | | | | |
| Q-04 | | | | |
| Q-05 | | | | |
| Q-06 | | | | |
| Q-07 | | | | |
| Q-08 | | | | |
| Q-09 | | | | |
| Q-10 | | | | |
| **hit@3 total** | | /10 | | /10 |

**Both retrievers agree:** which query, and why was it easy?

>

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

>

**Embeddings did better:** which query, and what could BM25 not match?

>

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

## C1 — the complete prompt, printed before it is sent

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | |
| Question | |
| Retrieved context | |
| **Total input** | |
| **`usage.prompt_tokens` from the response** | |

- Number of retrieved chunks:
- Tokens per chunk:
- Total context size (tokens / characters):

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>

**What exactly does retrieval add to the model's input?**

>

## C2 — three systems, six questions

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | | | |
| C-02 | | | |
| C-03 | | | |
| C-04 | | | |
| C-05 | | | |
| C-06 | | | |
| **agrees with `reference`** | /6 | /6 | /6 |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>

**2. When can BM25 perform better than embeddings?**

>

**3. When can embeddings perform better than BM25?**

>

**4. Why is cosine similarity used with embeddings?**

>

**5. What information is actually sent to an LLM in a RAG system?**

>

**6. Does the LLM search the original documents itself?**

>

**7. What happens if retrieval returns an irrelevant chunk?**

>

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>

**9. Is a vector database required to build a RAG system?**

>

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>

---
# Conclusion

>